# BYOL (2020)
---
[[paper]](https://arxiv.org/abs/2006.07733)<br>
BYOL = Bootstrap Your Own Latent

BYOL — это алгоритм Self-Supervised Learning для обучения визуальных представлений (Image Representations) без использования размеченных данных. Разработан командой DeepMind. В отличие от большинства подходов того времени, метод не требует использования Negative Samples для обучения.

__Idea:__ избавиться от необходимости подбора сложных стратегий формирования негативных пар (Contrastive Learning) за счет использования асимметричной архитектуры, где одна сеть предсказывает выход другой сети для разных View одного и того же изображения.

### Контекст
В Self-Supervised Learning основной проблемой является **Collapse** — ситуация, когда модель начинает выдавать одинаковый вектор (константу) для любого входного изображения, тем самым минимизируя Loss тривиальным способом. До появления BYOL это решалось через Contrastive Learning (отталкивание разных изображений друг от друга), что требовало огромных Batch Size или сложных Memory Banks.

### Альтернативы
* **SimCLR (2020)**: использует Contrastive Loss (NT-Xent). Требует очень больших батчей (4096+) и агрессивных аугментаций, чтобы иметь достаточно Negative Samples.
* **MoCo (2019)**: использует очередь (Queue) для хранения негативных примеров и Momentum Encoder, но все еще опирается на механизм различения "свой-чужой".
* **SwAV (2020)**: использует Online Clustering, где представления должны соответствовать "прототипам", что также является формой предотвращения коллапса через разделение признаков.

### Архитектура
BYOL состоит из двух взаимодействующих нейросетей: **Online Network** и **Target Network**.

1.  **Online Network**: состоит из Encoder (например, ResNet-50), Projector (MLP) и **Predictor** (дополнительный MLP-слой).
2.  **Target Network**: имеет ту же архитектуру, что и Online, но **не имеет Predictor**. Её веса являются Exponential Moving Average (EMA) от весов Online-сети.
3.  **Asymmetry**: именно наличие Predictor в Online-сети и использование EMA в Target-сети не дает модели свалиться в коллапс. Predictor делает задачу предсказания направленной (Online предсказывает Target, но не наоборот).

### Алгоритм обучения
1.  Для входного изображения $x$ создаются два разных View ($v$ и $v'$) с помощью стохастических аугментаций (Crop, Color Jitter, Blur и т.д.).
2.  $v$ подается в Online Network, на выходе получаем предсказание $q_\theta(z_v)$.
3.  $v'$ подается в Target Network, на выходе получаем проекцию $z'_{v'}$.
4.  **Loss Function**: вычисляется Mean Squared Error (MSE) между нормализованными векторами предсказания и таргета: $L = \| \bar{q}_\theta(z_v) - \bar{z}'_{v'} \|_2^2$.
5.  **Backpropagation**: градиенты рассчитываются и применяются **только для Online Network**.
6.  **Target Update**: веса Target-сети обновляются как $\xi \leftarrow \tau \xi + (1 - \tau) \theta$, где $\tau$ — коэффициент сглаживания (близкий к 1). Target-сеть выступает в роли "медленно меняющегося учителя".

### Алгоритм инференса
После завершения обучения Projector и Predictor отбрасываются. Для практических задач (Classification, Detection) используется только **Encoder** из Online Network в качестве Feature Extractor.

### Результаты
* **Устойчивость к батчам**: BYOL сохраняет высокую точность (68.3% Top-1 на ImageNet) даже при уменьшении Batch Size до 64, в то время как точность SimCLR падает с 66% до 50% из-за нехватки Negative Samples.
* **Производительность**: на архитектуре ResNet-50 (1x) BYOL достиг 74.3% Top-1 на ImageNet, что на 1.3 п.п. выше, чем у SimCLR, и вплотную приблизило Self-Supervised методы к результатам Fully Supervised моделей (76.5%).
* **Feature Robustness**: замена набора аугментаций (например, удаление Color Jitter) влияет на BYOL значительно меньше, чем на контрастивные методы, так как модель меньше полагается на простые цветовые признаки для различения объектов.

## 📝 Критический анализ

```markdown
# BYOL (2020)
---
[[paper]](https://arxiv.org/abs/2006.07733)<br>
BYOL = Bootstrap Your Own Latent

BYOL — это алгоритм **Self-Supervised Learning** для обучения визуальных представлений без размеченных данных, разработанный DeepMind. В отличие от других подходов, метод не требует Negative Samples.

__Идея:__ избавиться от сложных стратегий формирования негативных пар (Contrastive Learning) с помощью асимметричной архитектуры, где одна сеть предсказывает выход другой для разных View одного изображения.

### Контекст
В Self-Supervised Learning проблема **Collapse** возникает, когда модель выдает одинаковый вектор для любого изображения, минимизируя Loss тривиально. До BYOL это решалось через Contrastive Learning, требующий больших Batch Size или сложных Memory Banks.

### Альтернативы
* **SimCLR (2020)**: использует Contrastive Loss, требует больших батчей и агрессивных аугментаций.
* **MoCo (2019)**: использует очередь для хранения негативных примеров и Momentum Encoder.
* **SwAV (2020)**: применяет Online Clustering для предотвращения коллапса.

### Архитектура
BYOL состоит из **Online Network** и **Target Network**.

1. **Online Network**: включает Encoder (например, ResNet-50), Projector (MLP) и **Predictor** (дополнительный MLP).
2. **Target Network**: аналогична Online, но без Predictor. Веса обновляются как Exponential Moving Average (EMA) от весов Online.
3. **Асимметрия**: наличие Predictor и EMA предотвращает коллапс.

<img src="img/img.png" width=500>

### Алгоритм обучения
1. Создаются два View ($v$ и $v'$) изображения $x$ с помощью аугментаций.
2. $v$ проходит через Online Network, получаем предсказание $q_\theta(z_v)$.
3. $v'$ проходит через Target Network, получаем проекцию $z'_{v'}$.
4. **Loss Function**: Mean Squared Error между нормализованными векторами предсказания и таргета.
5. **Backpropagation**: градиенты применяются только к Online Network.
6. **Target Update**: веса Target обновляются как $\xi \leftarrow \tau \xi + (1 - \tau) \theta$.

### Алгоритм инференса
После обучения Projector и Predictor отбрасываются. Для задач используется только **Encoder** из Online Network как Feature Extractor.

### Результаты
* **Устойчивость к батчам**: BYOL сохраняет высокую точность (68.3% Top-1 на ImageNet) при Batch Size 64, в то время как SimCLR падает до 50%.
* **Производительность**: на ResNet-50 BYOL достиг 74.3% Top-1 на ImageNet, что на 1.3 п.п. выше SimCLR.
* **Feature Robustness**: BYOL менее зависим от цветовых признаков, чем контрастивные методы.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
from PIL import Image

# Dummy dataset for illustration
class DummyDataset(Dataset):
    def __init__(self, num_samples=100):
        self.num_samples = num_samples
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Generate a random image
        img = Image.fromarray(np.uint8(np.random.rand(256, 256, 3) * 255))
        return self.transform(img), self.transform(img)

# Define the BYOL architecture components
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        self.resnet = models.resnet50(pretrained=False)
        self.resnet.fc = nn.Identity()  # Remove the final classification layer

    def forward(self, x):
        return self.resnet(x)

class Projector(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=4096, output_dim=256):
        super(Projector, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

class Predictor(nn.Module):
    def __init__(self, input_dim=256, hidden_dim=4096, output_dim=256):
        super(Predictor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

# BYOL Model
class BYOL(nn.Module):
    def __init__(self):
        super(BYOL, self).__init__()
        self.online_encoder = Encoder()
        self.online_projector = Projector()
        self.predictor = Predictor()

        self.target_encoder = Encoder()
        self.target_projector = Projector()

        # Initialize target networks with the same weights as online networks
        self._initialize_target_networks()

    def _initialize_target_networks(self):
        for target_param, online_param in zip(self.target_encoder.parameters(), self.online_encoder.parameters()):
            target_param.data.copy_(online_param.data)
        for target_param, online_param in zip(self.target_projector.parameters(), self.online_projector.parameters()):
            target_param.data.copy_(online_param.data)

    def update_target_networks(self, tau=0.99):
        # Update target networks using exponential moving average
        for target_param, online_param in zip(self.target_encoder.parameters(), self.online_encoder.parameters()):
            target_param.data = tau * target_param.data + (1 - tau) * online_param.data
        for target_param, online_param in zip(self.target_projector.parameters(), self.online_projector.parameters()):
            target_param.data = tau * target_param.data + (1 - tau) * online_param.data

    def forward(self, x1, x2):
        # Online network forward pass
        online_proj1 = self.online_projector(self.online_encoder(x1))
        online_pred1 = self.predictor(online_proj1)

        # Target network forward pass
        with torch.no_grad():
            target_proj2 = self.target_projector(self.target_encoder(x2))

        return online_pred1, target_proj2

# Loss function
def byol_loss(pred, target):
    pred = nn.functional.normalize(pred, dim=-1)
    target = nn.functional.normalize(target, dim=-1)
    return 2 - 2 * (pred * target).sum(dim=-1)

# Training loop
def train_byol():
    dataset = DummyDataset()
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    model = BYOL()
    optimizer = optim.Adam(model.parameters(), lr=3e-4)

    for epoch in range(10):  # Dummy training loop
        for x1, x2 in dataloader:
            optimizer.zero_grad()

            pred, target = model(x1, x2)
            loss = byol_loss(pred, target).mean()

            loss.backward()
            optimizer.step()

            # Update target networks
            model.update_target_networks()

        print(f"Epoch {epoch}, Loss: {loss.item()}")

train_byol()
```

### Key Points:
- **Asymmetry**: The Online Network has a Predictor, while the Target Network does not. This asymmetry helps prevent collapse.
- **EMA Update**: The Target Network's weights are updated using an Exponential Moving Average of the Online Network's weights.
- **Loss Function**: The loss is computed as the Mean Squared Error between the normalized predictions of the Online Network and the projections of the Target Network.
- **No Negative Samples**: Unlike contrastive methods, BYOL does not rely on negative samples, simplifying the training process and reducing the need for large batch sizes.